# ARCHIVIST — one run, end to end

**Source:** [github.com/sahamgorengancuan-prog/shopify-automation](https://github.com/sahamgorengancuan-prog/shopify-automation) · branch
[`claude/apparel-design-pipeline-6e3g0q`](https://github.com/sahamgorengancuan-prog/shopify-automation/tree/claude/apparel-design-pipeline-6e3g0q) · [house system](https://github.com/sahamgorengancuan-prog/shopify-automation/tree/claude/apparel-design-pipeline-6e3g0q/archivist/house) ·
[discovery](https://github.com/sahamgorengancuan-prog/shopify-automation/tree/claude/apparel-design-pipeline-6e3g0q/archivist/discovery) ·
[open in Colab](https://colab.research.google.com/github/sahamgorengancuan-prog/shopify-automation/blob/claude/apparel-design-pipeline-6e3g0q/notebooks/archivist_pipeline.ipynb)

Fill in the parameter cell, then **Runtime → Run all**. This notebook runs *one* pipeline to
completion — market signal, creative route, one image, print proof, packaged delivery. It is a
runner, not an application: no UI is started here.

**What it costs.** One paid image generation by default. Everything the code can do for free is
done locally: the blueprint, background cleanup, placement, typography, colour separation, the
mockup and every measurement. A second paid call happens only if you explicitly allow one *and*
the failure class says money can fix it.

**What it refuses.** Nothing ships unless the measured proof passes. A failed candidate is packaged
as a diagnosis (`REJECTED_review.zip`) with every number that produced the verdict — never as a
"best of a bad batch" final.

In [ ]:
# @title 1. Parameters — this is the only cell you edit
# Leave TOPIC empty to let volume-first discovery choose the market signal.
TOPIC = ""  # @param {type:"string"}
AUDIENCE = "design-literate streetwear buyers who value quiet conceptual graphics"  # @param {type:"string"}
COLLECTION = "colab-run"  # @param {type:"string"}
GARMENT = "dark"  # @param ["dark", "faded-black", "black", "light", "white", "sand"]
AGGRESSIVENESS = 3  # @param {type:"slider", min:0, max:10, step:1}
TRENDS_GEO = "US"  # @param {type:"string"}

# --- house system -----------------------------------------------------------
ASYMMETRY_ANCHOR = "auto"  # @param ["auto", "upper-left", "upper-right", "low-left", "low-right"]
STATEMENT_OVERRIDE = ""  # @param {type:"string"}
PRINT_STATEMENT = True  # @param {type:"boolean"}

# --- spending ---------------------------------------------------------------
# 1 paid generation is the default. A second is only used when you allow it AND
# the failure is one a second call could actually fix.
PAID_GENERATION_BUDGET = 1  # @param {type:"slider", min:1, max:2, step:1}
ALLOW_CONTROLLED_EDIT = False  # @param {type:"boolean"}
ALLOW_CONCEPT_RETRY = False  # @param {type:"boolean"}
REQUIRE_VISION_CRITIC = True  # @param {type:"boolean"}
GENERATE_IMAGE = True  # @param {type:"boolean"}

# Recover an interrupted run for free: paste a raw frame path, or leave it blank
# and the setup cell will offer the newest one it finds.
REUSE_BFL_RAW = ""  # @param {type:"string"}

# --- behaviour --------------------------------------------------------------
USE_LLM = True  # @param {type:"boolean"}
OFFLINE = False  # @param {type:"boolean"}
RESET_STYLE_LOCK = True  # @param {type:"boolean"}
AUTO_DOWNLOAD = True  # @param {type:"boolean"}
SEED = 0  # @param {type:"integer"}

# --- keys -------------------------------------------------------------------
# Best practice: store keys in Colab Secrets (🔑 in the left sidebar) using these
# exact names, then leave the fields below empty. Anything typed here wins for
# this session only and is never written to disk unless you tick SAVE_KEYS_TO_ENV.
USE_COLAB_SECRETS = True  # @param {type:"boolean"}
SAVE_KEYS_TO_ENV = False  # @param {type:"boolean"}
BFL_API_KEY = ""  # @param {type:"string"}
OPENAI_API_KEY = ""  # @param {type:"string"}
PEXELS_API_KEY = ""  # @param {type:"string"}
REDDIT_CLIENT_ID = ""  # @param {type:"string"}
REDDIT_CLIENT_SECRET = ""  # @param {type:"string"}
X_BEARER_TOKEN = ""  # @param {type:"string"}
META_ACCESS_TOKEN = ""  # @param {type:"string"}
META_IG_USER_ID = ""  # @param {type:"string"}

print("Parameters loaded. Default paid-image budget:", PAID_GENERATION_BUDGET)

## How the run is decided

**1 · Market signal.** Broad one- and two-word roots are compared in Google Trends against one
shared benchmark (`archive` = 100) — the only way separate requests are comparable. Trends
throttles, so batches retry with backoff; if less than 80% of the pool answered, the ranking is
abandoned rather than chosen from holes. Only the leading quartile is measured in detail for
growth, competition and social heat.

**2 · Intent gate.** A high-volume word is not yet a concept. Each leader is checked for its
dominant public meaning; the run refuses to leap from a broad signal to an unmeasured specialist
system just because it sounds intellectual.

**3 · Creative route.** Evidence → one physical mutation → displaced composition → one printed
sentence. Three routes are produced (two authored, one audited fallback) and a skeptical jury
picks one; nothing weak wins by default.

**4 · Structural gate.** Runs *before* any spending: source property, single mutation, statement
rules, reference role contract, blueprint-only conditioning, and a nameable subject.

**5 · One generation.** Only an owned blueprint conditions the image model — searched references
inform the brief and never the pixels. The blueprint carries the subject's silhouette archetype,
which is what stops a strong composition from reading as generic rubble.

**6 · Proof.** Deterministic and free first: asymmetry, hero envelope, mode-aware ink coverage,
clean canvas edges, statement contrast and cap height. Only then does the visual critic look at
the rendered pixels, and it must clear 82/100 with no critical score under 8.

Full rules: [`archivist/house/rules.py`](https://github.com/sahamgorengancuan-prog/shopify-automation/blob/claude/apparel-design-pipeline-6e3g0q/archivist/house/rules.py) ·
[`silhouette.py`](https://github.com/sahamgorengancuan-prog/shopify-automation/blob/claude/apparel-design-pipeline-6e3g0q/archivist/house/silhouette.py) ·
[`proof.py`](https://github.com/sahamgorengancuan-prog/shopify-automation/blob/claude/apparel-design-pipeline-6e3g0q/archivist/house/proof.py)

In [ ]:
# @title 2. Source from GitHub, keys, and settings
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/sahamgorengancuan-prog/shopify-automation.git"
REPO_BRANCH = "claude/apparel-design-pipeline-6e3g0q"

# --- source: use a local checkout if the notebook is already inside one -------
ROOT = Path.cwd()
if not (ROOT / "archivist").is_dir():
    ROOT = ROOT.parent if (ROOT.parent / "archivist").is_dir() else None
if ROOT is None:
    ROOT = Path("/content/archivist-src") if Path("/content").is_dir() else Path.cwd() / "archivist-src"
    if (ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REPO_BRANCH],
                       capture_output=True, text=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", f"origin/{REPO_BRANCH}"],
                       capture_output=True, text=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(ROOT)],
                       check=True, capture_output=True, text=True)
sys.path.insert(0, str(ROOT))

from archivist import notebook as nb

# Installs only what a headless run needs — the UI stack is never downloaded here.
report = nb.bootstrap(ROOT, repo_url=REPO_URL, branch=REPO_BRANCH)
print(f"source: {report['source']} at {report['path']} ({report.get('commit', 'local')})")
if report["installed"]:
    print("installed:", ", ".join(report["installed"]))

# --- keys ---------------------------------------------------------------------
statuses = nb.load_secrets(
    {
        "BFL_API_KEY": BFL_API_KEY, "OPENAI_API_KEY": OPENAI_API_KEY,
        "PEXELS_API_KEY": PEXELS_API_KEY, "REDDIT_CLIENT_ID": REDDIT_CLIENT_ID,
        "REDDIT_CLIENT_SECRET": REDDIT_CLIENT_SECRET, "X_BEARER_TOKEN": X_BEARER_TOKEN,
        "META_ACCESS_TOKEN": META_ACCESS_TOKEN, "META_IG_USER_ID": META_IG_USER_ID,
    },
    use_colab_secrets=USE_COLAB_SECRETS, root=ROOT, persist=SAVE_KEYS_TO_ENV,
)

import os
os.environ.setdefault("ARCHIVIST_RUNS_DIR", str(Path("/content/archivist_runs") if Path("/content").is_dir() else ROOT / "runs"))

from archivist.config import Settings

settings = Settings.from_env(
    ROOT, collection=COLLECTION, garment=GARMENT, offline=OFFLINE, seed=SEED,
    trends_geo=TRENDS_GEO, house_anchor=ASYMMETRY_ANCHOR,
    house_statement_override=STATEMENT_OVERRIDE.strip(),
)

# Free recovery: an interrupted run has already been paid for.
recoverable = nb.find_recoverable_raw(settings.runs_dir)
if not REUSE_BFL_RAW.strip() and recoverable:
    print(f"\nfound an unfinished paid frame you can reuse for free:\n  REUSE_BFL_RAW = \"{recoverable}\"")

try:
    from IPython.display import Markdown, display
    display(Markdown(nb.secret_table(statuses)))
except ImportError:
    print(nb.secret_table(statuses))

print()
print(nb.capability_line(statuses, offline=OFFLINE))
print("runs directory:", Path(settings.runs_dir).resolve())

In [ ]:
# @title 3. Test every connection before spending
from archivist import connectivity

checks = connectivity.run_checks(settings, include_generation=GENERATE_IMAGE)
for check in checks:
    print(f"{check.badge:<12} {check.name:<28} {check.ms:>6} ms  {check.detail[:88]}")
print()
print(connectivity.summarise(checks))

## The single pipeline run

```text
market signal        → volume-first discovery, or the TOPIC you set
  → intent gate      → is this word designable at all?
  → creative route   → evidence, one mutation, one interruption, one sentence
  → reference audit  → research only; these pixels never reach the image model
  → structural gate  → the last free checkpoint before any spending
  → one generation   → conditioned on the owned blueprint
  → local finishing  → background, placement, typography, separation, mockup
  → proof            → measured checks, then the visual critic
  → delivery         → FINAL_* files, listing copy, placement spec, audit
```

If anything blocks the run, it stops **before** the paid call and says why.

In [ ]:
# @title 4. Run the pipeline once, to completion
from archivist.house import HouseBlocked, RenderOptions, run_session
from archivist.pipeline import PipelineOptions

options = PipelineOptions(
    audience=AUDIENCE, garment=GARMENT, aggressiveness=AGGRESSIVENESS, use_llm=USE_LLM,
)
render_options = RenderOptions(
    budget=PAID_GENERATION_BUDGET,
    allow_controlled_edit=ALLOW_CONTROLLED_EDIT,
    allow_concept_retry=ALLOW_CONCEPT_RETRY,
    require_critic=REQUIRE_VISION_CRITIC and USE_LLM and settings.can_use_llm,
    print_statement=PRINT_STATEMENT,
    reuse_raw=REUSE_BFL_RAW.strip(),
    seed=SEED,
)

result = None
try:
    result = run_session(
        settings, options,
        topic=TOPIC.strip(),
        render_options=render_options,
        generate=GENERATE_IMAGE,
        reset_style_lock=RESET_STYLE_LOCK,
        log=print,
    )
except HouseBlocked as blocked:
    print("\n" + "=" * 78)
    print("STOPPED BEFORE SPENDING ANYTHING")
    print(blocked)

if result is not None:
    print("\n" + "=" * 78)
    print("CREATIVE CONTRACT")
    for line in nb.contract_lines(result.route, result.discovery):
        print(" ", line)
    print()
    print("PROOF")
    for line in nb.proof_lines(result.delivery):
        print(" ", line)
    if result.rejected:
        print("\n  rejected — diagnosis package:", result.rejected)
    for warning in result.warnings:
        print("  warning:", warning)

In [ ]:
# @title 5. Preview the result and the evidence
from pathlib import Path

if result is None:
    print("Nothing to preview — the run stopped before producing artwork.")
else:
    try:
        from IPython.display import Image as IPyImage, Markdown, display
        inline = True
    except ImportError:
        inline = False

    for path, caption in nb.preview_images(result):
        print(caption)
        if inline:
            display(IPyImage(filename=path, width=430))
        else:
            print(" ", path)

    listing = Path(result.files.get("product_description", ""))
    if listing.is_file():
        text = listing.read_text(encoding="utf-8")
        if inline:
            display(Markdown(text))
        else:
            print(text)

In [ ]:
# @title 6. Package the result and download it
package = nb.package_delivery(result, output_dir=".") if result is not None else None

if package is None:
    print("No package: the run produced neither an approved delivery nor a rejection diagnosis.")
else:
    kind = "rejection diagnosis" if package.name.startswith("REJECTED") else "approved delivery"
    print(f"{kind}: {package}  ({package.stat().st_size / 1024:.0f} KB)")
    if AUTO_DOWNLOAD and nb.download(package):
        print("download started")
    elif AUTO_DOWNLOAD:
        print("(download is only automatic in Colab — copy the file from the path above)")

    if result.approved:
        print("\nBefore printing: order a physical sample and check the statement size, the colour")
        print("separation and the hand feel. Keep the supplied placement — never auto-centre it.")

---

**Run it again** with a different `TOPIC`, or leave it blank to let discovery pick the next signal.
Every run writes its full evidence — route, blueprint, measurements, critic verdict — into the run
directory, so a design can still answer *why this subject* months later.

Headless equivalents of this notebook:

```bash
python -m archivist house --topic harbor      # this notebook, on a terminal
python -m archivist volume                    # the demand ranking only
./scripts/linux/archivist.sh house            # Ubuntu launcher
```

There is also a local control room (`python -m archivist.app`) for browsing past runs and
scheduling — deliberately not started here, because this notebook is for one run.

Source: [github.com/sahamgorengancuan-prog/shopify-automation](https://github.com/sahamgorengancuan-prog/shopify-automation/tree/claude/apparel-design-pipeline-6e3g0q)